# Data Inspection And Sampling

In [3]:
from pathlib import Path
import pandas as pd

# Resolve project-relative paths no matter where the notebook is started from
cwd = Path.cwd()
project_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "data").exists()),
    cwd,
)

raw_dir = project_root / "data/raw"
interim_dir = project_root / "data/interim"
processed_dir = project_root / "data/processed"

# Create output directories so later `to_csv(...)` calls don't fail
interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

# Load dataset
df = pd.read_csv(raw_dir / "customer_support_tickets_200k.csv")

# Basic inspection
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False))

Shape: (200000, 30)

Columns:
['ticket_id', 'customer_name', 'customer_email', 'product', 'category', 'issue_description', 'resolution_notes', 'priority', 'status', 'channel', 'region', 'customer_age', 'customer_gender', 'subscription_type', 'customer_tenure_months', 'previous_tickets', 'customer_satisfaction_score', 'first_response_time_hours', 'resolution_time_hours', 'ticket_created_date', 'ticket_resolved_date', 'escalated', 'sla_breached', 'operating_system', 'browser', 'payment_method', 'language', 'preferred_contact_time', 'issue_complexity_score', 'customer_segment']

First 5 rows:
   ticket_id      customer_name                   customer_email  \
0          1     Patricia Smith    patricia.smith760@outlook.com   
1          2  Patricia Williams   patricia.williams390@gmail.com   
2          3   William Anderson  william.anderson651@outlook.com   
3          4       David Miller       david.miller672@icloud.com   
4          5    Robert Gonzalez   robert.gonzalez391@hotmail.co

### Sampling a manageable subset

In [4]:
# Sample 10K rows for faster experimentation
df_sample = df.sample(n=10000, random_state=42).copy()

print("Sample shape:", df_sample.shape)

Sample shape: (10000, 30)


### Data Cleaning

In [5]:
# Keep only rows that have issue + resolution text
df_sample = df_sample.dropna(subset=["issue_description", "resolution_notes"]).copy()

# Clean text
df_sample["issue_description"] = (
    df_sample["issue_description"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df_sample["resolution_notes"] = (
    df_sample["resolution_notes"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Remove very short noisy rows
df_sample = df_sample[df_sample["issue_description"].str.len() > 20]
df_sample = df_sample[df_sample["resolution_notes"].str.len() > 20]

# Remove duplicate issue descriptions
df_sample = df_sample.drop_duplicates(subset=["issue_description"]).copy()

print("Cleaned sample shape:", df_sample.shape)

Cleaned sample shape: (10, 30)


### Inspect the important project columns

In [6]:
important_cols = [
    "ticket_id",
    "product",
    "category",
    "issue_description",
    "resolution_notes",
    "priority",
    "status",
    "channel",
    "region",
    "subscription_type",
    "previous_tickets",
    "customer_satisfaction_score",
    "first_response_time_hours",
    "resolution_time_hours",
    "escalated",
    "sla_breached",
    "issue_complexity_score",
    "customer_segment"
]

print(df_sample[important_cols].head())

        ticket_id               product           category  \
119737     119738  Subscription Service         Bug Report   
72272       72273   Analytics Dashboard     Refund Request   
158154     158155         Cloud Storage     Refund Request   
65426       65427           API Service  Performance Issue   
30074       30075  Subscription Service         Bug Report   

                                        issue_description  \
119737  my subscription was cancelled without my reque...   
72272   i found a bug in the latest update affecting r...   
158154  i am unable to access my account after enterin...   
65426   the application crashes whenever i try to uplo...   
30074   two-factor authentication codes are not being ...   

                                         resolution_notes priority  \
119737  the duplicate charge was reversed and refund p...     High   
72272   explained billing breakdown and clarified appl...   Urgent   
158154  explained billing breakdown and clarified 

### Check value distributions

In [7]:
for col in ["priority", "category", "status", "channel", "region", "subscription_type", "escalated", "sla_breached"]:
    print(f"\n--- {col.upper()} ---")
    print(df_sample[col].value_counts(dropna=False).head(20))


--- PRIORITY ---
priority
High      3
Urgent    3
Medium    2
Low       2
Name: count, dtype: int64

--- CATEGORY ---
category
Bug Report            3
Refund Request        2
Performance Issue     2
Data Sync Issue       1
Security Concern      1
Account Suspension    1
Name: count, dtype: int64

--- STATUS ---
status
Pending Customer    4
Resolved            3
Open                2
Closed              1
Name: count, dtype: int64

--- CHANNEL ---
channel
Chat            3
Phone           2
Email           2
Web Form        2
Social Media    1
Name: count, dtype: int64

--- REGION ---
region
Asia             4
Europe           3
Africa           2
North America    1
Name: count, dtype: int64

--- SUBSCRIPTION_TYPE ---
subscription_type
Free          3
Basic         3
Premium       2
Enterprise    2
Name: count, dtype: int64

--- ESCALATED ---
escalated
Yes    6
No     4
Name: count, dtype: int64

--- SLA_BREACHED ---
sla_breached
No     6
Yes    4
Name: count, dtype: int64


### Check text examples

In [8]:
for i, row in df_sample[["issue_description", "resolution_notes", "priority", "category"]].head(5).iterrows():
    print(f"\nRow {i}")
    print("Issue:", row["issue_description"])
    print("Resolution:", row["resolution_notes"])
    print("Priority:", row["priority"])
    print("Category:", row["category"])


Row 119737
Issue: my subscription was cancelled without my request and i need clarification.
Resolution: the duplicate charge was reversed and refund processed within 3-5 business days.
Priority: High
Category: Bug Report

Row 72272
Issue: i found a bug in the latest update affecting report generation.
Resolution: explained billing breakdown and clarified applicable charges.
Priority: Urgent
Category: Refund Request

Row 158154
Issue: i am unable to access my account after entering the correct credentials.
Resolution: explained billing breakdown and clarified applicable charges.
Priority: High
Category: Refund Request

Row 65426
Issue: the application crashes whenever i try to upload a file.
Resolution: the duplicate charge was reversed and refund processed within 3-5 business days.
Priority: Medium
Category: Performance Issue

Row 30074
Issue: two-factor authentication codes are not being delivered to my phone.
Resolution: security settings updated and customer notified of precaution

### Save your cleaned inspection sample

In [10]:
from pathlib import Path

cwd = Path.cwd()
project_root = cwd
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

interim_dir = project_root / "data/interim"
interim_dir.mkdir(parents=True, exist_ok=True)

output_path = interim_dir / "tickets_sample_cleaned.csv"
df_sample.to_csv(output_path, index=False)
print(f"Saved cleaned sample to {output_path}")

Saved cleaned sample to c:\Users\akhil\Documents\Personal_Projects\Ops Decision Engine\data\interim\tickets_sample_cleaned.csv


##### ML Data Sample

In [11]:
ml_df = df_sample[[
    "issue_description",
    "priority",
    "category",
    "product",
    "channel",
    "issue_complexity_score",
    "previous_tickets",
    "escalated",
    "sla_breached"
]].copy()

ml_df.to_csv(processed_dir / "ml_priority_dataset.csv", index=False)
print("Saved ML dataset")

OSError: Cannot save file into a non-existent directory: 'data\processed'

#### RAG Data Sample

In [ ]:
rag_df = df_sample[[
    "ticket_id",
    "issue_description",
    "resolution_notes",
    "category",
    "product",
    "priority",
    "status"
]].copy()

rag_df.to_csv(processed_dir / "rag_incident_dataset.csv", index=False)
print("Saved RAG dataset")